# Categorical Semantic Extractor — FINAL V3 SELF-CONTAINED Dual-GPU Notebook

This version is corrected for the Kaggle input layout shown in your screenshot.

It expects only the **dataset** under `/kaggle/input/.../data/`; it does **not** expect `categorical_boolean_extractor/` to be present in the Kaggle input. The complete extractor/training architecture is embedded inside this notebook and is reconstructed under `/kaggle/working/version2`.

Protocol:

1. locate the four JSONL splits automatically,
2. reconstruct the complete code package,
3. verify 2 GPUs,
4. run final dataset QA + architecture tests,
5. freeze dataset hashes,
6. perform the **single heavy XLM-R training run** with 2-GPU DDP,
7. calibrate on VALIDATION only,
8. freeze/package before TEST,
9. evaluate frozen TEST once,
10. keep FINAL_HOLDOUT out of model training/calibration/inference,
11. create final downloadable artifacts.

## 1. Locate the dataset automatically

In [ ]:
from pathlib import Path
import os, sys, json, hashlib, shutil, re, unicodedata
from collections import Counter

INPUT_ROOT = Path('/kaggle/input')
PROJECT_ROOT = Path('/kaggle/working/version2')
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)

train_candidates = sorted(INPUT_ROOT.rglob('categorical_boolean_train.jsonl'))
if not train_candidates:
    raise FileNotFoundError(
        'categorical_boolean_train.jsonl was not found under /kaggle/input.'
    )

DATA_DIR = None
for train_path in train_candidates:
    candidate = train_path.parent
    required = [
        candidate / 'categorical_boolean_train.jsonl',
        candidate / 'categorical_boolean_validation.jsonl',
        candidate / 'categorical_boolean_test.jsonl',
        candidate / 'categorical_boolean_final_holdout.jsonl',
    ]
    if all(path.exists() for path in required):
        DATA_DIR = candidate
        break

if DATA_DIR is None:
    raise FileNotFoundError(
        'The four final JSONL split files were not found in the same data directory.'
    )

TRAIN = DATA_DIR / 'categorical_boolean_train.jsonl'
VALIDATION = DATA_DIR / 'categorical_boolean_validation.jsonl'
TEST = DATA_DIR / 'categorical_boolean_test.jsonl'
FINAL_HOLDOUT = DATA_DIR / 'categorical_boolean_final_holdout.jsonl'

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

print('DATA_DIR:', DATA_DIR)
print('PROJECT_ROOT:', PROJECT_ROOT)
for path in [TRAIN, VALIDATION, TEST, FINAL_HOLDOUT]:
    print(path.name, '->', round(path.stat().st_size/1024/1024, 2), 'MiB')


## 2. Reconstruct the COMPLETE architecture inside `/kaggle/working/version2`

In [ ]:
import base64
from zipfile import ZipFile

EMBEDDED_CODE_B64 = """..."""
EMBEDDED_CODE_ZIP = Path('/kaggle/working/categorical_boolean_embedded_code.zip')
EMBEDDED_CODE_ZIP.write_bytes(base64.b64decode(EMBEDDED_CODE_B64.encode('ascii')))

package_dir = PROJECT_ROOT / 'categorical_boolean_extractor'
if package_dir.exists():
    shutil.rmtree(package_dir)

with ZipFile(EMBEDDED_CODE_ZIP, 'r') as archive:
    archive.extractall(PROJECT_ROOT)

required_code = [
    package_dir / 'runtime.py',
    package_dir / 'semantic' / 'model.py',
    package_dir / 'training' / 'train_multitask_ddp.py',
    package_dir / 'training' / 'calibrate_confidence.py',
    package_dir / 'training' / 'evaluate_frozen.py',
    package_dir / 'training' / 'package_artifact.py',
]
for path in required_code:
    if not path.exists():
        raise FileNotFoundError(path)

print('Embedded architecture restored:', package_dir)
print('Python files:', len(list(package_dir.rglob('*.py'))))


## 3. Verify that Kaggle exposes BOTH GPUs

In [ ]:
import torch, subprocess

print('torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('Visible GPU count:', torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(f'GPU {i}:', torch.cuda.get_device_name(i))

if torch.cuda.device_count() < 2:
    raise RuntimeError(
        'This FINAL notebook requires a Kaggle accelerator with 2 GPUs. '
        'Enable a 2-GPU accelerator in Notebook Settings and restart the session.'
    )

subprocess.run(['nvidia-smi'], check=False)


## 4. Install/verify dependencies

In [ ]:
import subprocess

subprocess.run(
    [
        sys.executable, '-m', 'pip', 'install', '-q',
        'transformers>=4.45', 'numpy>=1.26', 'pytest>=8.0'
    ],
    check=True,
)

import transformers, numpy, pytest
print('transformers:', transformers.__version__)
print('numpy:', numpy.__version__)
print('pytest:', pytest.__version__)


## 5. FINAL dataset QA

This reruns the strict QA on the actual Kaggle files. It checks row counts, EN/FR/mixed balance, the full 4×4 HA/access matrix, stress-family presence, negative-class safety, and cross-split leakage contracts.

`FINAL_HOLDOUT` is read here only for structural QA; it is never passed to training, calibration, or model evaluation.

In [ ]:
!python -m categorical_boolean_extractor.training.qa_final_dataset \
  --data-dir "{DATA_DIR}"


## 6. Architecture tests before any heavy training

In [ ]:
!python -m pytest categorical_boolean_extractor/tests -q
!python -m categorical_boolean_extractor.architecture_smoke


## 7. Freeze immutable dataset hashes BEFORE training

In [ ]:
def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(1024*1024), b''):
            h.update(chunk)
    return h.hexdigest()

DATASET_HASHES = {
    'train': sha256_file(TRAIN),
    'validation': sha256_file(VALIDATION),
    'test': sha256_file(TEST),
    'final_holdout': sha256_file(FINAL_HOLDOUT),
}

PRETRAIN_FREEZE = {
    'status': 'READY_FOR_SINGLE_HEAVY_TRAINING_RUN',
    'dataset_hashes': DATASET_HASHES,
    'gpu_count': torch.cuda.device_count(),
    'gpu_names': [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
    'architecture': {
        'base_model': 'FacebookAI/xlm-roberta-base',
        'shared_encoder': True,
        'ha_classes': 4,
        'access_classes': 4,
        'loss': 'CE_HA + CE_ACCESS',
        'max_epochs': 4,
        'max_length': 128,
        'learning_rate': 2e-5,
        'weight_decay': 0.01,
        'checkpoint_metric': 'mean(HA_macro_F1, Access_macro_F1)',
    },
    'test_used': False,
    'final_holdout_used_for_modeling': False,
}

PRETRAIN_FREEZE_PATH = PROJECT_ROOT / 'PRETRAIN_FREEZE.json'
PRETRAIN_FREEZE_PATH.write_text(
    json.dumps(PRETRAIN_FREEZE, indent=2) + '\n', encoding='utf-8'
)
print(json.dumps(PRETRAIN_FREEZE, indent=2))


## 8. ONE heavy XLM-R training run on BOTH GPUs

This is the only heavy training cell. It uses:

```bash
torchrun --standalone --nproc_per_node=2
```

which launches one synchronized DDP process per GPU.

The best checkpoint is selected only on VALIDATION with:

`mean(HA macro-F1, Access macro-F1)`

**Do not rerun this cell after TEST.**

### Important V3 correction — DDP pooler bug fixed

The uploaded run reached the real 2-GPU training path, but DDP stopped because
XLM-R's `pooler.dense.weight` and `pooler.dense.bias` were trainable even though
this architecture classifies from `last_hidden_state[:, 0, :]` and never uses the
pooler.

V3 fixes the cause instead of enabling broad unused-parameter detection:

- the unused XLM-R pooler is explicitly frozen;
- `find_unused_parameters=False` remains strict for all genuinely trainable parameters;
- AdamW receives trainable parameters only;
- AMP uses the current `torch.amp` API;
- the LR scheduler advances only when GradScaler actually executes an optimizer step;
- a regression test verifies the frozen-pooler contract before training.

The failed run did not produce a completed checkpoint or TEST result, so rerunning
the heavy training cell after this code correction does **not** constitute model
fine-tuning or tuning from TEST.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

OUTPUT_DIR = PROJECT_ROOT / "categorical_boolean_xlmr_v1"
TRAINING_LOG = PROJECT_ROOT / "categorical_boolean_xlmr_v1_TRAINING.log"

# A completed run must never be overwritten.
if (OUTPUT_DIR / "training_metadata.json").exists():
    raise RuntimeError(
        "A COMPLETED heavy training run already exists. "
        "Do not retrain. Continue to the next cell."
    )

# If a previous torchrun failed before producing training_metadata.json,
# this is an incomplete run, not a valid frozen training result.
if OUTPUT_DIR.exists():
    existing = sorted(
        str(path.relative_to(OUTPUT_DIR))
        for path in OUTPUT_DIR.rglob("*")
        if path.is_file()
    )
    print("Removing incomplete previous training output:", OUTPUT_DIR)
    print("Incomplete files found:", existing[:30])
    shutil.rmtree(OUTPUT_DIR)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# PRE-FLIGHT BEFORE THE SINGLE HEAVY TRAINING RUN:
# Resolve/download XLM-R once now so both DDP ranks can use the shared HF cache.
print("Preflight: resolving FacebookAI/xlm-roberta-base ...")
from transformers import AutoModel, AutoTokenizer

_preflight_tokenizer = AutoTokenizer.from_pretrained(
    "FacebookAI/xlm-roberta-base"
)
_preflight_model = AutoModel.from_pretrained(
    "FacebookAI/xlm-roberta-base"
)
print("Preflight model/tokenizer: PASS")

del _preflight_model
del _preflight_tokenizer
if torch.cuda.is_available():
    torch.cuda.empty_cache()

env = os.environ.copy()
env["PYTHONPATH"] = (
    str(PROJECT_ROOT)
    + os.pathsep
    + env.get("PYTHONPATH", "")
)
env["TOKENIZERS_PARALLELISM"] = "false"
env["NCCL_DEBUG"] = "WARN"

command = [
    "torchrun",
    "--standalone",
    "--nproc_per_node=2",
    "-m",
    "categorical_boolean_extractor.training.train_multitask_ddp",
    "--train",
    str(TRAIN),
    "--validation",
    str(VALIDATION),
    "--output-dir",
    str(OUTPUT_DIR),
    "--base-model",
    "FacebookAI/xlm-roberta-base",
    "--epochs",
    "4",
    "--batch-size",
    "16",
    "--gradient-accumulation",
    "1",
    "--learning-rate",
    "2e-5",
    "--weight-decay",
    "0.01",
    "--max-length",
    "128",
    "--dropout",
    "0.1",
    "--seed",
    "20260826",
    "--num-workers",
    "2",
]

print("\nLaunching the ONE heavy 2-GPU DDP training run:")
print(" ".join(command))
print("\nFull output is also saved to:", TRAINING_LOG)

with TRAINING_LOG.open("w", encoding="utf-8") as log_file:
    process = subprocess.Popen(
        command,
        cwd=str(PROJECT_ROOT),
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    assert process.stdout is not None

    for line in process.stdout:
        print(line, end="")
        log_file.write(line)
        log_file.flush()

    return_code = process.wait()

if return_code != 0:
    tail = TRAINING_LOG.read_text(
        encoding="utf-8",
        errors="replace",
    ).splitlines()[-120:]

    raise RuntimeError(
        "The 2-GPU torchrun training FAILED with "
        f"exit code {return_code}.\n\n"
        "The metadata error in the following cell would only be a consequence.\n"
        "Last training-log lines:\n\n"
        + "\n".join(tail)
    )

required_after_training = [
    OUTPUT_DIR / "training_metadata.json",
    OUTPUT_DIR / "best_validation_metrics.json",
    OUTPUT_DIR / "training_history.json",
    OUTPUT_DIR / "model.pt",
    OUTPUT_DIR / "model_config.json",
    OUTPUT_DIR / "tokenizer",
    OUTPUT_DIR / "encoder_config",
]

missing_after_training = [
    str(path)
    for path in required_after_training
    if not path.exists()
]

if missing_after_training:
    raise RuntimeError(
        "torchrun exited with code 0, but required training artifacts "
        "are missing:\n" + "\n".join(missing_after_training)
    )

print("\nSTATUS: ONE_HEAVY_DUAL_GPU_TRAINING_COMPLETE")
print("OUTPUT_DIR:", OUTPUT_DIR)
print("TRAINING_LOG:", TRAINING_LOG)


## 9. Verify that the recorded training really used 2 GPUs

In [ ]:
required_training_files = [
    OUTPUT_DIR / "training_metadata.json",
    OUTPUT_DIR / "best_validation_metrics.json",
    OUTPUT_DIR / "training_history.json",
    OUTPUT_DIR / "model.pt",
    OUTPUT_DIR / "model_config.json",
]

missing = [
    str(path)
    for path in required_training_files
    if not path.exists()
]

if missing:
    log_hint = (
        "\nTraining log: " + str(TRAINING_LOG)
        if "TRAINING_LOG" in globals()
        else ""
    )
    raise RuntimeError(
        "The heavy training did not complete successfully. "
        "Do NOT continue to calibration/TEST.\nMissing:\n"
        + "\n".join(missing)
        + log_hint
    )

TRAINING_METADATA = json.loads(
    (OUTPUT_DIR / "training_metadata.json").read_text(
        encoding="utf-8"
    )
)
BEST_VALIDATION = json.loads(
    (OUTPUT_DIR / "best_validation_metrics.json").read_text(
        encoding="utf-8"
    )
)

print("TRAINING METADATA")
print(json.dumps(TRAINING_METADATA, indent=2))

print("\nBEST VALIDATION")
print(json.dumps(BEST_VALIDATION, indent=2))

if TRAINING_METADATA.get("world_size") != 2:
    raise RuntimeError(
        "Training completed but did not use the required two DDP processes. "
        f"world_size={TRAINING_METADATA.get('world_size')}"
    )

print("\nSTATUS: DUAL_GPU_TRAINING_METADATA_VERIFIED")


## 10. Calibrate confidence on VALIDATION only

The class-specific probability and margin thresholds are not hand-picked. The calibration code performs an **exact empirical two-dimensional search** over threshold values observed on validation predictions and maximizes accepted coverage subject to:

`accepted precision >= 0.99`

In [ ]:
!python -m categorical_boolean_extractor.training.calibrate_confidence \
  --checkpoint-dir "{OUTPUT_DIR}" \
  --validation "{VALIDATION}" \
  --target-precision 0.99 \
  --batch-size 32


## 11. Freeze/package model + calibration BEFORE TEST

In [ ]:
FROZEN_ARTIFACT = PROJECT_ROOT / 'categorical_boolean_xlmr_v1_FROZEN.zip'

!python -m categorical_boolean_extractor.training.package_artifact \
  --checkpoint-dir "{OUTPUT_DIR}" \
  --output "{FROZEN_ARTIFACT}"

FROZEN_BEFORE_TEST = {
    'status': 'MODEL_AND_CALIBRATION_FROZEN_BEFORE_TEST',
    'artifact': str(FROZEN_ARTIFACT),
    'artifact_sha256': sha256_file(FROZEN_ARTIFACT),
    'model_sha256': sha256_file(OUTPUT_DIR / 'model.pt'),
    'calibration_sha256': sha256_file(OUTPUT_DIR / 'calibration.json'),
    'dataset_hashes': DATASET_HASHES,
    'training_metadata': TRAINING_METADATA,
    'best_validation_metrics': BEST_VALIDATION,
    'test_seen': False,
    'final_holdout_seen_by_model': False,
    'rule': 'NO_MODEL_OR_CALIBRATION_MODIFICATION_BASED_ON_TEST',
}

FROZEN_BEFORE_TEST_PATH = PROJECT_ROOT / 'CATEGORICAL_BOOLEAN_FROZEN_BEFORE_TEST.json'
FROZEN_BEFORE_TEST_PATH.write_text(
    json.dumps(FROZEN_BEFORE_TEST, indent=2) + '\n', encoding='utf-8'
)
print(json.dumps(FROZEN_BEFORE_TEST, indent=2))


## 12. Frozen TEST — exactly once

This cell evaluates the already-frozen model and calibration. Do not change the model or calibration based on this result.

In [ ]:
TEST_SENTINEL = PROJECT_ROOT / 'FROZEN_TEST_ALREADY_EXECUTED.txt'
if TEST_SENTINEL.exists():
    raise RuntimeError(
        'Frozen TEST was already executed in this session/output. '
        'The guard prevents an accidental second TEST evaluation.'
    )

!python -m categorical_boolean_extractor.training.evaluate_frozen \
  --checkpoint-dir "{OUTPUT_DIR}" \
  --test "{TEST}" \
  --batch-size 32

TEST_SENTINEL.write_text('FROZEN TEST EXECUTED ONCE\n', encoding='utf-8')

FROZEN_TEST_METRICS = json.loads(
    (OUTPUT_DIR / 'frozen_test_metrics.json').read_text(encoding='utf-8')
)
print(json.dumps(FROZEN_TEST_METRICS, indent=2))


## 13. Frozen production integration smoke

In [ ]:
from categorical_boolean_extractor import (
    CategoricalBooleanExtractor,
    CategoricalBooleanLLMFallback,
)

extractor = CategoricalBooleanExtractor.from_artifact(
    FROZEN_ARTIFACT,
    expected_sha256=sha256_file(FROZEN_ARTIFACT),
    llm_fallback=CategoricalBooleanLLMFallback(enabled=False),
)

SMOKE_TEXTS = [
    'HA matters for the team, but no requirement has been decided. Clients perform I/O in parallel.',
    'The service must stay online through a node failure. Requests jump between unrelated offsets.',
    'High availability is not required. Jobs scan files from beginning to end.',
    'Bulk data is streamed in order while metadata uses scattered point lookups.',
]

PRODUCTION_SMOKE = []
for text in SMOKE_TEXTS:
    result = extractor.extract(text)
    PRODUCTION_SMOKE.append(result.to_dict())

PRODUCTION_SMOKE_PATH = PROJECT_ROOT / 'categorical_boolean_frozen_production_smoke.json'
PRODUCTION_SMOKE_PATH.write_text(
    json.dumps(PRODUCTION_SMOKE, indent=2, ensure_ascii=False) + '\n',
    encoding='utf-8',
)
print(json.dumps(PRODUCTION_SMOKE, indent=2, ensure_ascii=False))


## 14. Build the FINAL deliverable ZIP

The final deliverable contains the frozen model artifact, all freeze metadata, training/validation/test metrics, calibration, production smoke, and the exact embedded package used. FINAL_HOLDOUT remains unused by the model.

In [ ]:
from zipfile import ZIP_DEFLATED, ZipFile

FINAL_DELIVERABLE = PROJECT_ROOT / 'categorical_boolean_semantic_FINAL_DELIVERABLE.zip'
FINAL_METADATA_PATH = PROJECT_ROOT / 'categorical_boolean_FINAL_METADATA.json'

files_to_include = {
    FROZEN_ARTIFACT: FROZEN_ARTIFACT.name,
    PRETRAIN_FREEZE_PATH: PRETRAIN_FREEZE_PATH.name,
    FROZEN_BEFORE_TEST_PATH: FROZEN_BEFORE_TEST_PATH.name,
    OUTPUT_DIR / 'training_metadata.json': 'training_metadata.json',
    OUTPUT_DIR / 'training_history.json': 'training_history.json',
    OUTPUT_DIR / 'best_validation_metrics.json': 'best_validation_metrics.json',
    OUTPUT_DIR / 'calibration.json': 'calibration.json',
    OUTPUT_DIR / 'frozen_test_metrics.json': 'frozen_test_metrics.json',
    OUTPUT_DIR / 'model_config.json': 'model_config.json',
    PRODUCTION_SMOKE_PATH: PRODUCTION_SMOKE_PATH.name,
}

if FINAL_DELIVERABLE.exists():
    FINAL_DELIVERABLE.unlink()

with ZipFile(FINAL_DELIVERABLE, 'w', ZIP_DEFLATED) as archive:
    for source, archive_name in files_to_include.items():
        if not source.exists():
            raise FileNotFoundError(source)
        archive.write(source, archive_name)

    # Include exact source package used by the notebook.
    for source in sorted((PROJECT_ROOT / 'categorical_boolean_extractor').rglob('*.py')):
        archive.write(
            source,
            Path('source_code') / source.relative_to(PROJECT_ROOT),
        )

FINAL_METADATA = {
    'status': 'FINAL_DELIVERABLE_READY',
    'deliverable': str(FINAL_DELIVERABLE),
    'deliverable_sha256': sha256_file(FINAL_DELIVERABLE),
    'frozen_artifact': str(FROZEN_ARTIFACT),
    'frozen_artifact_sha256': sha256_file(FROZEN_ARTIFACT),
    'training_world_size': TRAINING_METADATA['world_size'],
    'best_validation_metrics': BEST_VALIDATION,
    'frozen_test_metrics': FROZEN_TEST_METRICS,
    'test_evaluated_once_after_freeze': True,
    'final_holdout_used_for_modeling': False,
}

FINAL_METADATA_PATH.write_text(
    json.dumps(FINAL_METADATA, indent=2) + '\n', encoding='utf-8'
)
print(json.dumps(FINAL_METADATA, indent=2))


## 15. DOWNLOAD — clickable links

In [ ]:
from IPython.display import FileLink, display

os.chdir(PROJECT_ROOT)

print('FINAL DELIVERABLE')
display(FileLink(FINAL_DELIVERABLE.name))

print('\nFROZEN MODEL ARTIFACT')
display(FileLink(FROZEN_ARTIFACT.name))

print('\nFINAL METADATA')
display(FileLink(FINAL_METADATA_PATH.name))

print('\nPaths:')
print(FINAL_DELIVERABLE)
print(FROZEN_ARTIFACT)
print(FINAL_METADATA_PATH)
